In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, confusion_matrix
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import mlflow
import mlflow.sklearn
import joblib
import logging
from pathlib import Path
import time
import json

# Configure display
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

print("✓ All imports successful!")

c:\Users\yezid\Desktop\mlops projects\mlops_env\Lib\site-packages\pydantic\_internal\_fields.py:149: UserWarning: Field "model_name" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


✓ All imports successful!


In [ ]:
print("\n" + "="*80)
print("STEP 1: LOAD AND PREPARE DATA")
print("="*80)

# ============================================================================
# LOAD THE DATA
# ============================================================================
# Read the CSV file we generated earlier
# This creates a DataFrame (like an Excel spreadsheet in Python)
df = pd.read_csv('../data/sample_logs/network_logs.csv')
#   '../data/' means go up one folder level, then into data/
#   This is because we're in notebooks/ so we need to go up

print(f"\n✓ Data loaded!")
print(f"  Shape: {df.shape[0]} rows × {df.shape[1]} columns")
# df.shape[0] = number of rows (10000 logs)
# df.shape[1] = number of columns (13 features like bytes_sent, packet_count, etc)

print(f"  Labels: {df['label'].value_counts().to_dict()}")
# This shows: how many 'normal' logs, how many 'port_scan', etc.
# Expected output:
# {'normal': 9500, 'port_scan': 500, 'ddos': 600, 'data_exfiltration': 400}

# ============================================================================
# SEPARATE NORMAL FROM ATTACK LOGS
# ============================================================================
# Why? Models learn from NORMAL data, then flag anything different
# So we need to clearly identify which logs are normal and which are attacks

normal_logs = df[df['label'] == 'normal'].copy()
# df['label'] == 'normal' creates a True/False list
# df[True/False list] selects only rows where label is 'normal'
# .copy() makes a copy so we don't accidentally modify the original

attack_logs = df[df['label'] != 'normal'].copy()
# df['label'] != 'normal' selects everything that is NOT normal
# This includes: port_scan, ddos, data_exfiltration

print(f"\nData split:")
print(f"  Normal: {len(normal_logs)} (95%)")
# len(normal_logs) counts how many normal logs we have
# Expected: 9500 normal logs (95% of 10000)

print(f"  Attacks: {len(attack_logs)} (5%)")
# Expected: 2500 attack logs (5% of 10000)

# ============================================================================
# WHAT THIS CELL TEACHES US
# ============================================================================
# ✓ Total dataset: 10,000 logs
# ✓ 95% normal (learning data)
# ✓ 5% attacks (testing data)
# ✓ Ready for feature extraction!


STEP 1: LOAD AND PREPARE DATA

✓ Data loaded!
  Shape: 10000 rows × 13 columns
  Labels: {'normal': 6000, 'port_scan': 1500, 'ddos': 1250, 'data_exfiltration': 1250}

Data split:
  Normal: 6000 (95%)
  Attacks: 4000 (5%)


In [ ]:
print("\n" + "="*80)
print("STEP 2: EXTRACT FEATURES")
print("="*80)

# ============================================================================
# DEFINE WHICH FEATURES TO USE
# ============================================================================
# A "feature" is a measurement/column we'll feed to the model
# We pick features that help distinguish normal from attack traffic

features_to_use = [
    'bytes_sent',          # How many bytes (data) were sent? (normal ~1000, attack ~50000+)
    'bytes_received',      # How many bytes were received? (normal ~1000, attack ~0)
    'packet_count',        # How many packets? (normal ~200, attack ~1, ~1000, ~500)
    'duration_seconds',    # How long did connection last? (normal ~5s, attack ~0.1s or ~30s)
    'source_port',         # What port did traffic come from? (varies)
    'dest_port'            # What port was traffic going to? (varies)
]

print(f"\n✓ Features extracted!")
print(f"  Number of features: {len(features_to_use)}")
print(f"  Features: {features_to_use}")

# ============================================================================
# CREATE FEATURE MATRIX (X) AND LABELS (y)
# ============================================================================
# X = Input data (the 6 measurements for each log)
# y = Output labels (0=normal, 1=attack)

X = df[features_to_use].copy()
# df[features_to_use] selects only the 6 columns we care about
# This creates a matrix: 10000 rows × 6 columns
# .copy() prevents accidental changes to original data

y = (df['label'] != 'normal').astype(int)
# df['label'] != 'normal' creates True/False: Is this an attack?
# .astype(int) converts: True→1 (attack), False→0 (normal)
# This is our "ground truth" - what the right answer is

print(f"\nFeature statistics:")
print(X.describe().round(2))
# X.describe() shows statistics for each feature:
# - count: how many values
# - mean: average value
# - std: how spread out values are
# - min/max: smallest and largest values
# This helps us understand the data

# ============================================================================
# SEPARATE NORMAL AND ATTACK FEATURES
# ============================================================================
# Why? Will use normal features to train the models
# Will use attack features to test if models detect them

X_normal = X[df['label'] == 'normal'].copy()
# All rows where label is 'normal'
# This is what the model will learn from
# Shape: (9500, 6) = 9500 normal logs with 6 features each

X_attack = X[df['label'] != 'normal'].copy()
# All rows where label is NOT normal (all attacks)
# This is what we'll test against
# Shape: (2500, 6) = 2500 attack logs with 6 features each

print(f"\n✓ Normal features: {X_normal.shape}")
print(f"✓ Attack features: {X_attack.shape}")

# ============================================================================
# WHAT THIS CELL TEACHES US
# ============================================================================
# ✓ Selected 6 most important features
# ✓ Created X (features) and y (labels)
# ✓ Split into normal (9500) and attack (2500)
# ✓ Ready for normalization!


STEP 2: EXTRACT FEATURES

✓ Features extracted!
  Number of features: 6
  Features: ['bytes_sent', 'bytes_received', 'packet_count', 'duration_seconds', 'source_port', 'dest_port']

Feature statistics:
       bytes_sent  bytes_received  packet_count  duration_seconds  \
count    10000.00        10000.00      10000.00          10000.00   
mean    131825.38         1442.70        345.16              6.84   
std     344949.97         1499.32        319.82             14.14   
min          0.00            0.00          1.00              0.00   
25%        736.12            0.00         87.75              0.50   
50%       1983.36         1115.99        272.00              2.19   
75%       9955.57         2229.01        458.00              6.84   
max    2221144.61        14803.95       1000.00            194.55   

       source_port  dest_port  
count     10000.00   10000.00  
mean      57308.74    5814.19  
std        4753.91   13733.23  
min       49153.00      22.00  
25%       53188

In [ ]:
print("\n" + "="*80)
print("STEP 3: NORMALIZE FEATURES")
print("="*80)

# ============================================================================
# WHAT IS NORMALIZATION?
# ============================================================================
# Problem: Our features have VERY different scales
# - bytes_sent: 0 to 2,000,000 (HUGE range!)
# - packet_count: 1 to 5,000 (tiny range)
# - duration_seconds: 0.1 to 120 (medium range)
#
# Solution: Normalize = Scale everything to -3 to +3 range
# Formula: (value - mean) / standard_deviation
# This puts all features on the SAME scale

# ============================================================================
# CREATE SCALER
# ============================================================================
scaler = StandardScaler()
# StandardScaler is a tool that:
# 1. Calculates mean and standard deviation of each feature
# 2. Transforms data: (value - mean) / std
# 3. Result: mean=0, std=1 (range: -3 to +3)

# ============================================================================
# FIT AND TRANSFORM ALL DATA
# ============================================================================
X_normalized = scaler.fit_transform(X)
# .fit_transform() does TWO things:
#   1. fit() = Learn the statistics (mean, std) from X
#   2. transform() = Apply normalization to X
# Result: X_normalized has same shape as X but scaled to -3 to +3

X_normalized_normal = scaler.transform(X_normal)
# .transform() applies the LEARNED scaling to X_normal
# Uses same mean/std from scaler (don't relearn!)
# Shape: (9500, 6) but all values now -3 to +3

X_normalized_attack = scaler.transform(X_attack)
# Apply same scaling to X_attack
# Important: Use the SAME scaler learned from X!
# This way all data uses consistent scaling
# Shape: (2500, 6) all normalized

print(f"\n✓ Features normalized!")

# ============================================================================
# VERIFY NORMALIZATION WORKED
# ============================================================================
print(f"\nBefore normalization:")
print(f"  bytes_sent range: {X['bytes_sent'].min():.0f} to {X['bytes_sent'].max():.0f}")
# Before: bytes_sent ranges from 0 to 2,000,000 (huge!)

print(f"  packet_count range: {X['packet_count'].min():.0f} to {X['packet_count'].max():.0f}")
# Before: packet_count ranges from 1 to 5,000

print(f"\nAfter normalization:")
print(f"  bytes_sent range: {X_normalized[:, 0].min():.2f} to {X_normalized[:, 0].max():.2f}")
# After: bytes_sent now ranges from ~-0.45 to ~575 (more reasonable!)
# X_normalized[:, 0] means: all rows, column 0 (bytes_sent)

print(f"  packet_count range: {X_normalized[:, 2].min():.2f} to {X_normalized[:, 2].max():.2f}")
# After: packet_count now ranges from ~-0.89 to ~14.92
# X_normalized[:, 2] means: all rows, column 2 (packet_count)

print(f"\n✓ All features now on scale: -3 to +3")
print(f"  X_normalized shape: {X_normalized.shape}")
# Shape confirms: same 10000 rows, 6 columns, just scaled values

print(f"  X_normalized_normal shape: {X_normalized_normal.shape}")
# 9500 normal logs, scaled

# ============================================================================
# WHY THIS MATTERS FOR MODELS
# ============================================================================
# Models like:
# 1. Isolation Forest: Works better with normalized data
# 2. Autoencoder: Neural networks NEED normalized data (-3 to +3)
#
# Without normalization:
# - Model would think bytes_sent is "more important" (bigger numbers)
# - Would give wrong weights to features
# - Would make bad predictions
#
# With normalization:
# - All features treated equally
# - Model learns true patterns, not just big numbers

# ============================================================================
# WHAT THIS CELL TEACHES US
# ============================================================================
# ✓ Normalized data: -3 to +3 range
# ✓ Same scaler used for all data (important!)
# ✓ Normal data: (9500, 6)
# ✓ Ready for model training!


STEP 3: NORMALIZE FEATURES

✓ Features normalized!

Before normalization:
  bytes_sent range: 0 to 2221145
  packet_count range: 1 to 1000

After normalization:
  bytes_sent range: -0.38 to 6.06
  packet_count range: -1.08 to 2.05

✓ All features now on scale: -3 to +3
  X_normalized shape: (10000, 6)
  X_normalized_normal shape: (6000, 6)


In [ ]:
print("\n" + "="*80)
print("DATA QUALITY CHECK")
print("="*80)

# Check what labels we actually have
print("\nLabel distribution in dataset:")
print(df['label'].value_counts())

# Check data types
print("\nData types:")
print(df.dtypes)

# Check for missing values
print("\nMissing values:")
print(df.isnull().sum())

# Check feature ranges BEFORE normalization
print("\n\nFeature ranges (BEFORE normalization):")
for col in features_to_use:
    print(f"  {col:20s}: min={X[col].min():.2f}, max={X[col].max():.2f}, mean={X[col].mean():.2f}")

# Check if features have variance (different values)
print("\n\nFeature variance:")
for col in features_to_use:
    print(f"  {col:20s}: std={X[col].std():.2f}")
    # std=0 means all values are same (no variance, model can't learn!)

# Check normalized data
print("\n\nFeature ranges (AFTER normalization):")
for i, col in enumerate(features_to_use):
    print(f"  {col:20s}: min={X_normalized[:, i].min():.2f}, max={X_normalized[:, i].max():.2f}")


DATA QUALITY CHECK

Label distribution in dataset:
label
normal               6000
port_scan            1500
ddos                 1250
data_exfiltration    1250
Name: count, dtype: int64

Data types:
timestamp            object
source_ip            object
dest_ip              object
source_port           int64
dest_port             int64
protocol             object
bytes_sent          float64
bytes_received      float64
packet_count          int64
duration_seconds    float64
flags                object
is_encrypted           bool
label                object
dtype: object

Missing values:
timestamp           0
source_ip           0
dest_ip             0
source_port         0
dest_port           0
protocol            0
bytes_sent          0
bytes_received      0
packet_count        0
duration_seconds    0
flags               0
is_encrypted        0
label               0
dtype: int64


Feature ranges (BEFORE normalization):
  bytes_sent          : min=0.00, max=2221144.61, mean=131825.38

In [ ]:
print("\n" + "="*80)
print("STEP 4: TRAIN ISOLATION FOREST")
print("="*80)

# ============================================================================
# WHAT IS ISOLATION FOREST?
# ============================================================================
# Algorithm: Randomly split features to separate "weird" data
# Analogy: Finding a needle in a haystack by randomly cutting the haystack
#   - Normal points (hay): Take many cuts to isolate
#   - Anomaly points (needle): Take few cuts to isolate
#
# Advantages:
#   ✓ Fast (seconds, not minutes)
#   ✓ Works well for obvious anomalies
#   ✓ No training needed on labeled data
#
# Disadvantages:
#   ✗ May miss subtle/novel attacks
#   ✗ Works best for clear outliers

# ============================================================================
# START MLFLOW EXPERIMENT
# ============================================================================
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("threat_detection")
# Create an MLflow "experiment" named "threat_detection"
# Experiments let us track and compare multiple model runs
# Like having a lab notebook for all our experiments

with mlflow.start_run(run_name="isolation_forest_v1"):
    # Start recording this run
    # run_name = "isolation_forest_v1" helps us identify this later
    # All logged metrics/params go into this run
    
    logger.info("\n1. Training Isolation Forest...")
    print("\nTraining Isolation Forest...")
    
    start_time = time.time()
    # Record start time (to measure how long training takes)
    
    # ====================================================================
    # CREATE AND TRAIN MODEL
    # ====================================================================
    iso_forest = IsolationForest(
        contamination=0.05,  # Expected % of anomalies (we set to 5%)
                             # Model will flag top 5% as anomalies
        
        random_state=42,     # For reproducibility
                             # Same seed = same results every run
        
        n_estimators=100,    # Number of trees to build
                             # More trees = better but slower
        
        n_jobs=-1            # Use all CPU cores
                             # -1 means "use all available cores"
    )
    
    iso_forest.fit(X_normalized_normal)
    # .fit() trains the model on NORMAL data only
    # Model learns: "What does normal traffic look like?"
    # Will later flag anything different as anomaly
    
    training_time = time.time() - start_time
    print(f"✓ Training complete in {training_time:.1f}s!")
    # Display how fast it was (should be ~1-3 seconds)
    
    # ====================================================================
    # MAKE PREDICTIONS
    # ====================================================================
    predictions_iso = iso_forest.predict(X_normalized)
    # .predict() returns:
    #   +1 = Normal (within learned pattern)
    #   -1 = Anomaly (outside learned pattern)
    
    predictions_iso_binary = (predictions_iso == -1).astype(int)
    # Convert to standard format: 0=normal, 1=anomaly
    # (predictions_iso == -1) creates True/False list
    # .astype(int) converts True→1, False→0
    
    anomaly_scores_iso = iso_forest.score_samples(X_normalized)
    # score_samples() returns "how anomalous" each point is
    # More negative = more anomalous
    # Less negative = more normal
    # Used for AUC-ROC calculation
    
    print("✓ Predictions complete!")
    
    # ====================================================================
    # EVALUATE MODEL
    # ====================================================================
    print("\nEvaluating performance...")
    
    y_true = (df['label'] != 'normal').astype(int)
    # Get ground truth labels: 0=normal, 1=attack
    # These are the CORRECT answers we compare against
    
    # Calculate metrics
    auc = roc_auc_score(y_true, -anomaly_scores_iso)
    # AUC-ROC: Overall accuracy (0 to 1)
    # 0.5 = random guessing, 1.0 = perfect
    # How well does the model separate normal from attack?
    
    precision = precision_score(y_true, predictions_iso_binary)
    # Precision: Of detected threats, how many real?
    # precision = TP / (TP + FP)
    # If we say "attack!" how often are we right?
    # High precision = fewer false alarms
    
    recall = recall_score(y_true, predictions_iso_binary)
    # Recall: Of actual threats, how many detected?
    # recall = TP / (TP + FN)
    # Do we catch all the attacks?
    # High recall = fewer missed attacks
    
    f1 = f1_score(y_true, predictions_iso_binary)
    # F1-Score: Balance between precision and recall
    # Harmonic mean of precision and recall (0 to 1)
    # Best of both worlds
    
    tn, fp, fn, tp = confusion_matrix(y_true, predictions_iso_binary).ravel()
    # Confusion matrix breakdown:
    # TP = True Positives (correctly detected attacks)
    # FP = False Positives (incorrectly flagged normal as attack)
    # TN = True Negatives (correctly identified normal)
    # FN = False Negatives (missed attacks - worst case!)
    
    # ====================================================================
    # DISPLAY RESULTS
    # ====================================================================
    print(f"\n{'='*80}")
    print(f"ISOLATION FOREST RESULTS")
    print(f"{'='*80}")
    print(f"\nMetrics:")
    print(f"  AUC-ROC:    {auc:.4f}")
    print(f"  Precision:  {precision:.4f}")
    print(f"  Recall:     {recall:.4f}")
    print(f"  F1-Score:   {f1:.4f}")
    
    print(f"\nConfusion Matrix:")
    print(f"  True Positives (caught attacks):    {tp}")
    # Correctly identified as attacks ✓
    
    print(f"  False Positives (false alarms):    {fp}")
    # Incorrectly flagged normal as attack (annoying but not dangerous)
    
    print(f"  True Negatives (correct normal):   {tn}")
    # Correctly identified as normal ✓
    
    print(f"  False Negatives (missed attacks):  {fn}")
    # Failed to detect actual attacks (very bad!) ✗
    
    # ====================================================================
    # LOG TO MLFLOW
    # ====================================================================
    # Save results so we can compare with other models later
    mlflow.log_param("model", "Isolation Forest")
    mlflow.log_param("contamination", 0.05)
    mlflow.log_param("n_estimators", 100)
    # log_param() = save settings/hyperparameters
    
    mlflow.log_metric("auc_roc", auc)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)
    # log_metric() = save performance scores
    
    print(f"\n✓ Logged to MLflow!")

# ============================================================================
# SAVE FOR COMPARISON
# ============================================================================
# Store metrics so we can compare with Autoencoder later
iso_auc = auc
iso_precision = precision
iso_recall = recall
iso_f1 = f1

# ============================================================================
# WHAT THIS CELL TEACHES US
# ============================================================================
# ✓ Isolation Forest trained (3 seconds!)
# ✓ AUC-ROC: How well does it separate normal/attack?
# ✓ Precision: How many false alarms?
# ✓ Recall: How many attacks missed?
# ✓ Results logged to MLflow
# ✓ Ready to compare with Autoencoder!


STEP 4: TRAIN ISOLATION FOREST


2026/06/28 00:06:12 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/28 00:06:12 INFO mlflow.store.db.utils: Updating database tables
2026/06/28 00:06:14 INFO mlflow.tracking.fluent: Experiment with name 'threat_detection' does not exist. Creating a new experiment.
INFO:__main__:
1. Training Isolation Forest...



Training Isolation Forest...
✓ Training complete in 0.3s!
✓ Predictions complete!

Evaluating performance...

ISOLATION FOREST RESULTS

Metrics:
  AUC-ROC:    0.9847
  Precision:  0.9287
  Recall:     0.9765
  F1-Score:   0.9520

Confusion Matrix:
  True Positives (caught attacks):    3906
  False Positives (false alarms):    300
  True Negatives (correct normal):   5700
  False Negatives (missed attacks):  94

✓ Logged to MLflow!


In [ ]:
print("\n" + "="*80)
print("STEP 5: TRAIN AUTOENCODER")
print("="*80)

# ============================================================================
# WHAT IS AN AUTOENCODER?
# ============================================================================
# Neural network that learns normal patterns by compression:
# Input (6) → Compress to 2 → Expand back to 6
# 
# If reconstruction error is HIGH → ANOMALY
# If reconstruction error is LOW → NORMAL
# 
# For detailed explanation, see previous notes above

with mlflow.start_run(run_name="autoencoder_v1"):
    print("\n1. Building Autoencoder architecture...")
    
    input_dim = X_normalized_normal.shape[1]  # 6 features
    
    # Encoder: 6 → 4 → 2 (compression)
    encoder_input = keras.Input(shape=(input_dim,))
    encoded = layers.Dense(4, activation='relu')(encoder_input)
    encoded = layers.Dense(2, activation='relu')(encoded)
    
    # Decoder: 2 → 4 → 6 (decompression)
    decoded = layers.Dense(4, activation='relu')(encoded)
    decoded = layers.Dense(input_dim, activation='linear')(decoded)
    
    # Full model
    autoencoder = keras.Model(encoder_input, decoded)
    autoencoder.compile(optimizer='adam', loss='mse')
    
    print(f"✓ Model built!")
    print(f"\nArchitecture:")
    print(f"  Input: {input_dim} features")
    print(f"  Encoder: 6 → 4 → 2")
    print(f"  Decoder: 2 → 4 → 6")
    
    # Train on normal logs only
    print(f"\n2. Training on {len(X_normalized_normal)} normal logs...")
    print(f"   (This takes ~3 minutes)\n")
    
    start_time = time.time()
    
    history = autoencoder.fit(
        X_normalized_normal,        # Input
        X_normalized_normal,        # Target (same - self-supervised)
        epochs=50,
        batch_size=32,
        validation_split=0.2,
        verbose=1
    )
    
    training_time = time.time() - start_time
    print(f"\n✓ Training complete in {training_time:.1f}s!")
    
    # Compute reconstruction errors
    print("\n3. Computing reconstruction errors...")
    reconstructed = autoencoder.predict(X_normalized, verbose=0)
    reconstruction_errors = np.mean(np.abs(reconstructed - X_normalized), axis=1)
    print("✓ Reconstruction errors computed!")
    
    # Set threshold
    print("\n4. Setting anomaly threshold...")
    normal_errors = reconstruction_errors[:len(X_normalized_normal)]
    threshold = np.percentile(normal_errors, 95)
    
    print(f"\nNormal reconstruction error stats:")
    print(f"  Mean: {normal_errors.mean():.6f}")
    print(f"  Std:  {normal_errors.std():.6f}")
    print(f"  95th percentile (threshold): {threshold:.6f}")
    
    # Make predictions
    predictions_ae = (reconstruction_errors > threshold).astype(int)
    
    # Evaluate
    print(f"\n5. Evaluating...")
    
    y_true = (df['label'] != 'normal').astype(int)
    
    auc = roc_auc_score(y_true, -reconstruction_errors)  # Negate for proper AUC
    precision = precision_score(y_true, predictions_ae)
    recall = recall_score(y_true, predictions_ae)
    f1 = f1_score(y_true, predictions_ae)
    tn, fp, fn, tp = confusion_matrix(y_true, predictions_ae).ravel()
    
    print(f"\n{'='*80}")
    print(f"AUTOENCODER RESULTS")
    print(f"{'='*80}")
    print(f"\nMetrics:")
    print(f"  AUC-ROC:    {auc:.4f}")
    print(f"  Precision:  {precision:.4f}")
    print(f"  Recall:     {recall:.4f}")
    print(f"  F1-Score:   {f1:.4f}")
    
    print(f"\nConfusion Matrix:")
    print(f"  True Positives (caught attacks):    {tp}")
    print(f"  False Positives (false alarms):    {fp}")
    print(f"  True Negatives (correct normal):   {tn}")
    print(f"  False Negatives (missed attacks):  {fn}")
    
    # Log to MLflow
    mlflow.log_param("model", "Autoencoder")
    mlflow.log_param("encoding_dim", 2)
    mlflow.log_param("epochs", 50)
    mlflow.log_metric("auc_roc", auc)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)
    
    print(f"\n✓ Logged to MLflow!")

# Store for comparison
ae_auc = auc
ae_precision = precision
ae_recall = recall
ae_f1 = f1

print("\n" + "="*80)
print("✓ AUTOENCODER TRAINING COMPLETE!")
print("="*80)


STEP 5: TRAIN AUTOENCODER

1. Building Autoencoder architecture...


✓ Model built!

Architecture:
  Input: 6 features
  Encoder: 6 → 4 → 2
  Decoder: 2 → 4 → 6

2. Training on 6000 normal logs...
   (This takes ~3 minutes)

Epoch 1/50
150/150 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.3827 - val_loss: 0.3342
Epoch 2/50
150/150 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.2873 - val_loss: 0.2622
Epoch 3/50
150/150 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.2375 - val_loss: 0.2067
Epoch 4/50
150/150 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.1663 - val_loss: 0.1290
Epoch 5/50
150/150 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.1096 - val_loss: 0.0925
Epoch 6/50
150/150 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0850 - val_loss: 0.0775
Epoch 7/50
150/150 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0737 - val_loss: 0.0693
Epoch 8/50
150/150 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0674 - val_loss: 0.0649
Epoch 9/50
150/150 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0639 - val_loss: 0.0623
Epoch 10/50
150/150 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.0620 - 

In [ ]:
print("\n" + "="*80)
print("STEP 6: MODEL COMPARISON")
print("="*80)

# ============================================================================
# CREATE COMPARISON TABLE
# ============================================================================
# Side-by-side comparison of both models
comparison_df = pd.DataFrame({
    'Metric': ['AUC-ROC', 'Precision', 'Recall', 'F1-Score'],
    'Isolation Forest': [iso_auc, iso_precision, iso_recall, iso_f1],
    'Autoencoder': [ae_auc, ae_precision, ae_recall, ae_f1]
})

print("\n" + comparison_df.to_string(index=False))

# ============================================================================
# DETERMINE WINNER
# ============================================================================
print("\n" + "="*80)
print("ANALYSIS")
print("="*80)

if ae_f1 > iso_f1:
    winner = "Autoencoder"
    winner_f1 = ae_f1
    loser_f1 = iso_f1
else:
    winner = "Isolation Forest"
    winner_f1 = iso_f1
    loser_f1 = ae_f1

print(f"\n✓ {winner} WINS!")
print(f"  F1-Score: {winner_f1:.4f} vs {loser_f1:.4f}")

print(f"\n{'='*80}")
print("RECOMMENDATION FOR PRODUCTION")
print(f"{'='*80}")

print(f"\nDeploy: {winner}")
print(f"\nWhy:")
print(f"  - Better F1-Score ({winner_f1:.4f})")
print(f"  - Better balance of precision and recall")
print(f"  - Catches more attacks with fewer false alarms")

if winner == "Autoencoder":
    print(f"\n  Speed: Slightly slower (~10ms per prediction)")
    print(f"  Accuracy: Excellent (~97%+)")
else:
    print(f"\n  Speed: Very fast (~1ms per prediction)")
    print(f"  Accuracy: Excellent (~95%+)")

print(f"\n" + "="*80)
print("DETAILED COMPARISON")
print(f"{'='*80}")

metrics = ['AUC-ROC', 'Precision', 'Recall', 'F1-Score']
iso_scores = [iso_auc, iso_precision, iso_recall, iso_f1]
ae_scores = [ae_auc, ae_precision, ae_recall, ae_f1]

for metric, iso_score, ae_score in zip(metrics, iso_scores, ae_scores):
    diff = ae_score - iso_score
    winner_symbol = "✓" if ae_score > iso_score else ("✗" if iso_score > ae_score else "=")
    print(f"\n{metric}:")
    print(f"  Isolation Forest: {iso_score:.4f}")
    print(f"  Autoencoder:      {ae_score:.4f}")
    if diff != 0:
        print(f"  Difference:       {abs(diff):+.4f} {winner_symbol}")

print(f"\n" + "="*80)
print("✓ COMPARISON COMPLETE - Ready to save models!")
print(f"{'='*80}")


STEP 6: MODEL COMPARISON

   Metric  Isolation Forest  Autoencoder
  AUC-ROC          0.984667     0.002527
Precision          0.928673     1.000000
   Recall          0.976500     0.120000
 F1-Score          0.951986     0.214286

ANALYSIS

✓ Isolation Forest WINS!
  F1-Score: 0.9520 vs 0.2143

RECOMMENDATION FOR PRODUCTION

Deploy: Isolation Forest

Why:
  - Better F1-Score (0.9520)
  - Better balance of precision and recall
  - Catches more attacks with fewer false alarms

  Speed: Very fast (~1ms per prediction)
  Accuracy: Excellent (~95%+)

DETAILED COMPARISON

AUC-ROC:
  Isolation Forest: 0.9847
  Autoencoder:      0.0025
  Difference:       +0.9821 ✗

Precision:
  Isolation Forest: 0.9287
  Autoencoder:      1.0000
  Difference:       +0.0713 ✓

Recall:
  Isolation Forest: 0.9765
  Autoencoder:      0.1200
  Difference:       +0.8565 ✗

F1-Score:
  Isolation Forest: 0.9520
  Autoencoder:      0.2143
  Difference:       +0.7377 ✗

✓ COMPARISON COMPLETE - Ready to save models!


In [ ]:
print("\n" + "="*80)
print("STEP 7: SAVE MODELS AND ARTIFACTS")
print("="*80)

# ============================================================================
# CREATE MODELS DIRECTORY
# ============================================================================
Path('../models').mkdir(exist_ok=True)
print("\n1. Saving Isolation Forest...")

# ============================================================================
# SAVE ISOLATION FOREST
# ============================================================================
joblib.dump(iso_forest, '../models/isolation_forest.pkl')
print("✓ Isolation Forest saved to: ../models/isolation_forest.pkl")

print("\n2. Saving Autoencoder...")

# ============================================================================
# SAVE AUTOENCODER
# ============================================================================
autoencoder.save('../models/autoencoder.h5')
print("✓ Autoencoder saved to: ../models/autoencoder.h5")

print("\n3. Saving Scaler...")

# ============================================================================
# SAVE SCALER
# ============================================================================
joblib.dump(scaler, '../models/scaler.pkl')
print("✓ Scaler saved to: ../models/scaler.pkl")

print("\n4. Saving metadata...")

# ============================================================================
# SAVE METADATA
# ============================================================================
feature_metadata = {
    'features': features_to_use,
    'n_features': len(features_to_use),
    'iso_forest_auc': float(iso_auc),
    'iso_forest_f1': float(iso_f1),
    'autoencoder_auc': float(ae_auc),
    'autoencoder_f1': float(ae_f1),
    'ae_threshold': float(threshold),
    'dataset_size': len(df),
    'training_notes': 'Isolation Forest trained on 10000 logs: 60% normal, 40% attacks'
}

with open('../models/metadata.json', 'w') as f:
    json.dump(feature_metadata, f, indent=2)
print("✓ Metadata saved to: ../models/metadata.json")

# ============================================================================
# LIST ALL SAVED FILES
# ============================================================================
print("\n" + "="*80)
print("SAVED ARTIFACTS SUMMARY")
print("="*80)

import os
models_path = '../models'

total_size = 0
if os.path.exists(models_path):
    for file in sorted(os.listdir(models_path)):
        file_path = os.path.join(models_path, file)
        file_size = os.path.getsize(file_path) / 1024
        total_size += file_size
        print(f"  ✓ {file:30s} ({file_size:>7.1f} KB)")

print(f"\n  Total size: {total_size:.1f} KB")

# ============================================================================
# FINAL SUMMARY
# ============================================================================
print("\n" + "="*80)
print("✓ TRAINING AND SAVING COMPLETE!")
print("="*80)

print(f"\n📊 FINAL RESULTS (ISOLATION FOREST):")
print(f"  AUC-ROC:    {iso_auc:.4f} ✓")
print(f"  Precision:  {iso_precision:.4f} ✓")
print(f"  Recall:     {iso_recall:.4f} ✓")
print(f"  F1-Score:   {iso_f1:.4f} ✓")

print(f"\n📊 CAUGHT:")
print(f"  Attacks detected: 3906 out of 4000 (97.65%)")
print(f"  False alarms: 300 out of 6000 (5%)")

print(f"\n📁 MODELS SAVED:")
print(f"  ✓ isolation_forest.pkl (production model)")
print(f"  ✓ scaler.pkl (preprocessing)")
print(f"  ✓ metadata.json (configuration)")

print(f"\n🚀 NEXT STEPS:")
print(f"  1. Build FastAPI server")
print(f"  2. Load isolation_forest.pkl")
print(f"  3. Create /predict endpoint")
print(f"  4. Test the API")
print(f"  5. Deploy to production!")

print(f"\n" + "="*80)
print("✓ Model ready for API deployment!")
print("="*80)


STEP 7: SAVE MODELS AND ARTIFACTS

1. Saving Isolation Forest...
✓ Isolation Forest saved to: ../models/isolation_forest.pkl

2. Saving Autoencoder...
✓ Autoencoder saved to: ../models/autoencoder.h5

3. Saving Scaler...
✓ Scaler saved to: ../models/scaler.pkl

4. Saving metadata...
✓ Metadata saved to: ../models/metadata.json

SAVED ARTIFACTS SUMMARY
  ✓ autoencoder.h5                 (   34.3 KB)
  ✓ isolation_forest.pkl           ( 1421.8 KB)
  ✓ metadata.json                  (    0.5 KB)
  ✓ scaler.pkl                     (    1.0 KB)

  Total size: 1457.6 KB

✓ TRAINING AND SAVING COMPLETE!

📊 FINAL RESULTS (ISOLATION FOREST):
  AUC-ROC:    0.9847 ✓
  Precision:  0.9287 ✓
  Recall:     0.9765 ✓
  F1-Score:   0.9520 ✓

📊 CAUGHT:
  Attacks detected: 3906 out of 4000 (97.65%)
  False alarms: 300 out of 6000 (5%)

📁 MODELS SAVED:
  ✓ isolation_forest.pkl (production model)
  ✓ scaler.pkl (preprocessing)
  ✓ metadata.json (configuration)

🚀 NEXT STEPS:
  1. Build FastAPI server
  2. 